In [3]:
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_links, fetch_website_contents

In [4]:
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key Exists and begins with: {openai_api_key[:8]}")
else:
    print("OpenAI API Key does not exist.")

if anthropic_api_key:
    print(f"Anthropic API Key Exists and begins with: {anthropic_api_key[:8]}")
else:
    print("Anthropic API Key does not exist.")

openai = OpenAI()

claude = OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1")
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

OpenAI API Key Exists and begins with: sk-proj-
Anthropic API Key Exists and begins with: sk-ant-a


In [5]:
def stream_gpt(input_prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": input_prompt}
    ]

    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or "" #If content is empty or None, use an empty string instead.
        yield result

In [6]:
def stream_Claude(input_prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}, 
        {"role": "user", "content": input_prompt}
        ]
    stream = claude.chat.completions.create(
        model="claude-sonnet-4-5-20250929",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or "" #If content is empty or None, use an empty string instead.
        yield result

In [7]:
def stream_ollama(input_prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}, 
        {"role": "user", "content": input_prompt}
        ]
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or "" #If content is empty or None, use an empty string instead.
        yield result

In [8]:

# Again this is typical Experimental mindset - I'm changing the global variable we used above:

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [16]:
def stream_brochure(url, model):
    yield ""
    prompt = f"Please generate a company brochure for {url}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model == "GPT":
        result= stream_gpt(prompt)
    elif model == "Claude":
        result = stream_Claude(prompt)
    elif model == "Ollama":
        result = stream_ollama(prompt)
    else:        raise ValueError("Invalid model name. Choose from 'GPT', 'Claude', or 'Ollama'.")
    yield from result

In [18]:
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Claude", "Ollama"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["https://huggingface.co", "GPT"],
            ["https://edwarddonner.com", "Claude"],
            ["https://huggingface.co", "Ollama"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/maazbaig/projects/llm_engineering/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/Users/maazbaig/projects/llm_engineering/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/maazbaig/projects/llm_engineering/.venv/lib/python3.12/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/Users/maazbaig/projects/llm_engineering/.venv/lib/python3.12/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/maazbaig/projects/llm_engineering/.venv/lib/python3.12/site-packages/httpcore/_sync/connection.py", line 101, in handle_request
    raise exc
  File "/Users/

In [19]:
gr.close_all()

Closing server running on port: 7864
Closing server running on port: 7860
Closing server running on port: 7862
Closing server running on port: 7865
Closing server running on port: 7861
Closing server running on port: 7863
